Erdos Summer 2026 Project: NBA Head-to-Head Prediction Probability
GAM model 

In [33]:
#Created by Marshall Scott for Erdos Summer 2026
#Last update: July 2026

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, precision_score

pd.set_option('display.max_columns', None)
prefix = "../data/processed/"

#Suppressing np error warnings, which have cropped up. The model always converges.
np.seterr(over='ignore', divide='ignore', invalid='ignore')

{'divide': 'ignore', 'over': 'ignore', 'under': 'ignore', 'invalid': 'ignore'}

Data and feature generation

In [34]:
#Regular games that have roll 10 aversges and home-away matchups and differences
df_m   = (pd.read_csv(prefix + "TeamStatisticsFrom2010RegMatchups.csv")).dropna()
py_fea = ['fieldGoalsPercentage_roll10_diff', 'plusMinusPoints_roll10_diff', 'away_plusMinusPoints_roll10', 
          'win_roll10_diff', 'home_turnovers_roll10']

Train and Test Split

In [35]:
#Initial Test and Train split
xg_tr, xg_te, yg_tr, yg_te = train_test_split(df_m, df_m['win'], random_state=17, test_size=0.2)

PyGAM Initialization

In [36]:
#PyGAM initialization 
import scipy.sparse
from pygam import LogisticGAM, s, l, te

def to_array(self):
    return self.toarray()

scipy.sparse.spmatrix.A = property(to_array)

#Pipeline for pygam
pipe_g = Pipeline(steps=[
    ('scale', StandardScaler()),
    ('gam', LogisticGAM(
        s(0, 100, lam=10) + l(1) + l(2) + s(3, lam=10) + l(4) + te(0,1, lam=10), max_iter=300 
        ))
])


KFold Validation with 5 splits

In [37]:
#KFold Validation
#Loop that iterates over each column in the above list, fits, and calculates the accuracy

kf       = KFold(n_splits=5)
stat_ary = np.zeros((3, 5))

for i, (train_ix, test_ix) in enumerate(kf.split(xg_tr)):
   xr = xg_tr[py_fea].iloc[train_ix].values
   yr = yg_tr.iloc[train_ix]
   xh = xg_tr[py_fea].iloc[test_ix].values
   yh = yg_tr.iloc[test_ix]
   pipe_g.fit(xr,yr)

   y_pred = pipe_g['gam'].predict(xh)
   acc    = 1.0*sum(y_pred==yh) / len(y_pred)
   prob   = pipe_g['gam'].predict_proba(xh)
   roc    = roc_auc_score(yh, prob)
   prec   = precision_score(yh, y_pred)

   stat_ary[0][i] = acc
   stat_ary[1][i] = roc
   stat_ary[2][i] = prec

print('KFold Results\nAccuracy  Mean(STD) %.3f(%.3f)\nROC       Mean(STD) %.3f(%.3f)\nPrecision Mean(STD) %.3f(%.3f)'%(stat_ary[0].mean(), stat_ary[0].std(), stat_ary[1].mean(), stat_ary[1].std(),stat_ary[2].mean(), stat_ary[2].std()))



KFold Results
Accuracy  Mean(STD) 0.630(0.010)
ROC       Mean(STD) 0.679(0.010)
Precision Mean(STD) 0.701(0.007)


Full Model Results, Classification Report, and Season by season comparision 

In [38]:
#Full model test
pipe_g.fit(xg_tr[py_fea],yg_tr)

y_pred = pipe_g['gam'].predict(xg_te[py_fea])
acc    = 1.0*sum(y_pred==yg_te) / len(y_pred)
prob   = pipe_g['gam'].predict_proba(xg_te[py_fea])
roc    = roc_auc_score(yg_te, prob)
prec   = precision_score(yg_te, y_pred)

print('Full Model Results\nAccuracy  %.3f\nROC       %.3f\nPrecision %.3f'%(acc, roc, prec))

Full Model Results
Accuracy  0.626
ROC       0.671
Precision 0.693


In [39]:
#Classification report
#The win precision is significantly better than the lose precision
print('Classification Report')
print(classification_report(yg_tr, pipe_g['gam'].predict(xg_tr[py_fea]), target_names=['Loose', 'Win'], digits=3))

Classification Report
              precision    recall  f1-score   support

       Loose      0.556     0.646     0.598      6520
         Win      0.702     0.617     0.657      8784

    accuracy                          0.630     15304
   macro avg      0.629     0.632     0.627     15304
weighted avg      0.640     0.630     0.632     15304



In [40]:
#Season by season results
stat_ary = np.zeros((3,11))
j = 0
for i in range(2015, 2026):
    df_m_tt = df_m[df_m["season"].isin(range(i - 5, i))]
    df_m_val = df_m[df_m["season"] == i]
    X_tt = df_m_tt[py_fea]
    y_tt = df_m_tt['win']
    X_val = df_m_val[py_fea]
    y_val = df_m_val['win']

    pipe_g.fit(X_tt, y_tt)
    y_pred = pipe_g['gam'].predict(X_val)
    acc    = 1.0*sum(y_pred==y_val) / len(y_pred)
    prob   = pipe_g['gam'].predict_proba(X_val)
    roc    = roc_auc_score(y_val, prob)
    prec   = precision_score(y_val, y_pred)

    stat_ary[0][j] = acc
    stat_ary[1][j] = roc
    stat_ary[2][j] = prec
    j+=1

In [41]:
#Result dataframe construction and printing
yr = [x for x in range(2015,2026)]
df_comp = pd.DataFrame(stat_ary.T, index=yr, columns=['Accuracy', 'ROC', 'Precision'])

#Printing all results
print('Season by Season Results\n',df_comp)
print('\n\nMeans')
print(df_comp.mean())
print('\n\nStandard Deviations')
print(df_comp.std())



Season by Season Results
       Accuracy       ROC  Precision
2015  0.654472  0.706587   0.712660
2016  0.608943  0.655899   0.680365
2017  0.621951  0.675635   0.693878
2018  0.620325  0.675179   0.724315
2019  0.604396  0.658372   0.676533
2020  0.601852  0.652436   0.665962
2021  0.617886  0.656979   0.664463
2022  0.573171  0.618421   0.672131
2023  0.630894  0.690258   0.687063
2024  0.653878  0.704637   0.708621
2025  0.643379  0.712897   0.708904


Means
Accuracy     0.621013
ROC          0.673391
Precision    0.690445
dtype: float64


Standard Deviations
Accuracy     0.024262
ROC          0.028624
Precision    0.020591
dtype: float64
